In [1]:
import numpy as np
 
X = np.load('../data/X.npy')
y = np.load('../data/y.npy')

In [2]:
import torch
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

print(X.shape, y.shape)

torch.Size([28709, 224, 224, 3]) torch.Size([28709])


In [ ]:
X = X.permute(0,3,1,2) # (N, H, W, C) -> (N, C, H, W)
print(X.shape)

torch.Size([28709, 3, 224, 224])


In [4]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X,y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [5]:
import torch.nn as nn

class EmotionCnn(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3,322,3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3),
            nn.ReLU(),
            nn.MaxPool2d(2),

        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*54*54,64),
            nn.ReLU(),
            nn.Linear(64,7)
        )

    def forward(self,x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [ ]:
model = EmotionCnn()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(2):
    for batch_X, batch_y in loader:
        pred = model(batch_X)
        loss = loss_fn(pred, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")